In [ ]:
from server.pipeline.chunk_and_embed import DocumentIngestChunkEmbedPipeline
from logging_config import setup_logging
from dotenv import load_dotenv

load_dotenv(override=True)

setup_logging()

import logging
import os

from dotenv import load_dotenv

load_dotenv(override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

pipeline = DocumentIngestChunkEmbedPipeline(
    json_input_path=(
        "server/ingested_docs/"
        "Warehouse_SOPs_DTAC_2020_og.json"
    ),
    document_output_path=(
        "server/ingested_docs/llamaindex/"
        "Warehouse_SOPs_DTAC_2020_documents_enriched.json"
    ),
    node_output_path=(
        "server/ingested_docs/llamaindex/"
        "Warehouse_SOPs_DTAC_2020_nodes_enriched.json"
    ),
    module="DTAC",
    system="WMS",
    doc_type="sop",
    llm_model="gpt-4.1-mini",
    embedding_model="text-embedding-3-small",
    chunk_size=1200,
    chunk_overlap=200,
    next_page_context_chars=2000,
    title_nodes=5,
    questions_per_chunk=3,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

pipeline.run()

In [ ]:
from config.settings import settings
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from llama_index.llms.openai import OpenAI

retriever = settings.index.as_retriever(
    vector_store_query_mode=VectorStoreQueryMode.HYBRID,
    similarity_top_k=3,
)

llm = OpenAI(
    model="gpt-4.1-mini",
    api_key=settings.openai_api_key.get_secret_value(),
)

question = "how to add a sku to a location"

retrieved_nodes = retriever.retrieve(question)

context = "\n\n".join(
    (
        f"[Source {index} | "
        f"pages={item.node.metadata.get('covered_pages')}]\n"
        f"{item.node.get_content()}"
    )
    for index, item in enumerate(retrieved_nodes, start=1)
)

prompt = f"""
Answer the question using only the supplied SOP context with all fields.

Question:
{question}

Context:
{context}

Answer:
""".strip()

response = llm.complete(prompt)

print(response.text)